<a href="https://colab.research.google.com/github/shaan-byte/python_ds_colab/blob/main/End_to_End_Data_Fetching_Pipeline_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 4.3 — End-to-End Data Fetching Pipeline

**Module 1: Programming & Data Foundations**  
**Week 4 | Session 4.3**

---

## What we will cover today

1. What a data pipeline is — and what can go wrong
2. The naive pipeline — simple, but fragile
3. Fix 1 — proper error handling
4. Fix 2 — rate limiting
5. Fix 3 — parsing API responses into CSV and JSON
6. The finished pipeline — all fixes combined


---
## Setup — Run this first

In [1]:
import requests
import json
import csv
import time
import os
from datetime import datetime

print("All imports successful.")
print("requests version:", requests.__version__)

All imports successful.
requests version: 2.32.4


In [ ]:
# Enter your https://www.weatherapi.com/

---
# Part 1 — What is a Data Pipeline?

## The definition

A **data pipeline** is a sequence of steps that moves data from a source to a destination, transforming it along the way.

The source could be an API, a database, a file, a sensor, or a web page. The destination could be a CSV file, a database, a dashboard, or another API. The transformation is everything in between — cleaning, filtering, reshaping, validating.

Today's pipeline is specific:

```
[ API (OpenWeatherMap) ]  -->  [ Python script ]  -->  [ CSV file / JSON file ]
       Source                   Transform & fetch          Destination
```

---

## The scenario

You work for a logistics company that operates across 15 Indian cities. Every morning, a manager needs a weather report for all 15 cities to plan deliveries. Your job is to write a script that:

1. Fetches live weather data for all 15 cities
2. Saves the results as a CSV file (for the manager's spreadsheet)
3. Also saves as a JSON file (for the company's internal dashboard system)

This sounds simple. Let's write the first version and see how quickly it falls apart.

---

## Whiteboard: what could go wrong?

Before writing any code, let's think about every possible failure in this pipeline:

- What if one city name is spelled wrong?
- What if the internet connection drops halfway through?
- What if the API key expires?
- What if we make too many requests too fast and get rate-limited?
- What if the API is down entirely?
- What if the response structure is slightly different for some cities?

Each of these will crash the naive version. We will fix them one by one.

---
# Part 2 — The Naive Pipeline

Here is the simplest possible version of our pipeline. Read it carefully before running it.

In [7]:
# Our list of 15 cities — notice anything suspicious?

CITIES = [
    "Mumbai",
    "Kanpur",
    "Bangalore",
    "Chennai",
    "Kolkata",
    "Hyderabad",
    "Pune",
    "Ahmedabad",
    "Jaipur",
    "Lucknow",
    "Kochi",
    "Chandigarh",
    "Bhopal",
    "Coimbatoreee",    # <-- deliberate typo
    "Nagpur"
]

BASE_URL = "http://api.weatherapi.com/v1/current.json"

print(f"Cities to fetch: {len(CITIES)}")

Cities to fetch: 15


In [8]:
# THE NAIVE PIPELINE
# Simple, direct, no safety net.
# Run this and watch what happens.

results = []

for city in CITIES:
    response = requests.get(BASE_URL, params={"q": city, "key": "11098527892e4fa8ab390858262806"})
    data     = response.json()             # assumes the response is always valid JSON

    print(data)

    if response.status_code == 200:
      record = {
          "city":        data["location"]["name"],       # assumes these keys always exist
          "temp_c":      data["current"]["temp_c"],
          "humidity":    data["current"]["humidity"],
          "description": data["current"]['condition']["text"]
      }
      results.append(record)
      print(f"  Fetched: {city}")
    elif response.status_code == 429:
      time.sleep(1)

print(f"\nDone. {len(results)} records collected.")

{'location': {'name': 'Mumbai', 'region': 'Maharashtra', 'country': 'India', 'lat': 18.975, 'lon': 72.826, 'tz_id': 'Asia/Kolkata', 'localtime_epoch': 1782638822, 'localtime': '2026-06-28 14:57'}, 'current': {'last_updated_epoch': 1782638100, 'last_updated': '2026-06-28 14:45', 'temp_c': 31.2, 'temp_f': 88.2, 'is_day': 1, 'condition': {'text': 'Mist', 'icon': '//cdn.weatherapi.com/weather/64x64/day/143.png', 'code': 1030}, 'wind_mph': 14.8, 'wind_kph': 23.8, 'wind_degree': 233, 'wind_dir': 'SW', 'pressure_mb': 1004.0, 'pressure_in': 29.65, 'precip_mm': 0.05, 'precip_in': 0.0, 'humidity': 71, 'cloud': 50, 'feelslike_c': 38.2, 'feelslike_f': 100.7, 'windchill_c': 29.1, 'windchill_f': 84.4, 'heatindex_c': 33.7, 'heatindex_f': 92.7, 'dewpoint_c': 23.9, 'dewpoint_f': 75.0, 'vis_km': 3.5, 'vis_miles': 2.0, 'uv': 9.2, 'gust_mph': 19.0, 'gust_kph': 30.5, 'will_it_rain': 0, 'chance_of_rain': 16, 'will_it_snow': 0, 'chance_of_snow': 0, 'short_rad': 883.8, 'diff_rad': 196.42, 'dni': 0.0, 'gti': 0

### What just happened?

The pipeline crashed on `"Coimbatorr"` — the typo. The API returned a 404, and the response body was an error message, not weather data. When the code tried to access `data["name"]`, Python threw a `KeyError` because there is no `"name"` key in an error response.

The problem is not just the crash itself. The problem is:
- The 13 records we collected before the crash were never saved anywhere
- We have no idea which city failed or why
- The entire pipeline has to be re-run from the beginning

Let's look at what the actual error response looks like:

In [10]:
# What does the API actually return for a bad city name?

bad_response = requests.get(BASE_URL, params={"q": "Coimbatorr", "key": ":11098527892e4fa8ab390858262806",})

print(f"Status code: {bad_response.status_code}")
print(f"Response body:")
print(json.dumps(bad_response.json(), indent=2))
print()
print("There is no 'name', 'main', or 'weather' key here.")
print("That is why data['name'] threw a KeyError.")

Status code: 401
Response body:
{
  "error": {
    "code": 2006,
    "message": "API key is invalid."
  }
}

There is no 'name', 'main', or 'weather' key here.
That is why data['name'] threw a KeyError.


---
# Part 3 — Fix 1: Proper Error Handling

## The principle

A robust pipeline never assumes a request will succeed. It expects failures and handles them gracefully. The goal is:

1. A failed city should be **logged** with a clear message
2. A failed city should be **skipped** — not crash the whole run
3. The successful results should be **saved immediately** — not held in memory until the end

We need to handle two categories of failure:

**Category A: HTTP errors** — the request reached the server but the server said no (4xx, 5xx)

**Category B: Network errors** — the request never reached the server at all (no internet, DNS failure, timeout)

Python's `requests` library raises different exceptions for each:

| Situation | Exception raised |
|---|---|
| No internet / DNS failure | `requests.exceptions.ConnectionError` |
| Server took too long | `requests.exceptions.Timeout` |
| Any HTTP 4xx or 5xx (after calling raise_for_status) | `requests.exceptions.HTTPError` |
| Any of the above | `requests.exceptions.RequestException` (catches all) |

---

## The logging pattern

A professional pipeline keeps a log of everything — successes, failures, timestamps, reasons. Even a simple list of dictionaries works well for learning. In production, you would write to a log file.

In [ ]:
# A robust fetch function — handles all failure modes

def fetch_weather(city, api_key):
    """
    Fetches weather for a single city.

    Returns a tuple: (record_or_None, error_or_None)
    Exactly one of the two will be None:
      - Success: (record_dict, None)
      - Failure: (None, error_message_string)
    """
    try:
        response = requests.get(
            BASE_URL,
            params  = {"q": city, "appid": api_key, "units": "metric"},
            timeout = 10    # give up if no response in 10 seconds
        )
        response.raise_for_status()    # raises HTTPError for 4xx / 5xx

        d = response.json()
        record = {
            "city":        d["name"],
            "country":     d["sys"]["country"],
            "temp_c":      d["main"]["temp"],
            "feels_like":  d["main"]["feels_like"],
            "humidity":    d["main"]["humidity"],
            "wind_ms":     d["wind"]["speed"],
            "description": d["weather"][0]["description"],
            "fetched_at":  datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
        }
        return record, None    # success

    except requests.exceptions.HTTPError as e:
        # Server responded but with an error code
        code = e.response.status_code if e.response else "?"
        if code == 404:
            return None, f"City not found (404): '{city}'"
        elif code == 401:
            return None, f"Invalid API key (401) — check your key"
        elif code == 429:
            return None, f"Rate limited (429) — too many requests"
        else:
            return None, f"HTTP error {code} for '{city}'"

    except requests.exceptions.Timeout:
        return None, f"Timeout: '{city}' took too long to respond"

    except requests.exceptions.ConnectionError:
        return None, f"Connection error: could not reach the server for '{city}'"

    except requests.exceptions.RequestException as e:
        return None, f"Unexpected network error for '{city}': {e}"


# Test: one good city, one bad city
record, error = fetch_weather("Mumbai", API_KEY)
print("Mumbai result:")
if record:
    print(f"  Success: {record['city']}, {record['temp_c']}C")
else:
    print(f"  Error: {error}")

print()

record2, error2 = fetch_weather("Coimbatorr", API_KEY)
print("Coimbatorr result:")
if record2:
    print(f"  Success: {record2['city']}, {record2['temp_c']}C")
else:
    print(f"  Error: {error2}")

In [ ]:
# Pipeline v2: with error handling
# Now a failure in one city no longer crashes the whole run

results_v2 = []
errors_v2  = []

for city in CITIES:
    record, error = fetch_weather(city, API_KEY)

    if record:
        results_v2.append(record)
        print(f"  OK    {city:<15} {record['temp_c']}C, {record['description']}")
    else:
        errors_v2.append({"city": city, "reason": error})
        print(f"  FAIL  {city:<15} {error}")

print()
print(f"Successful: {len(results_v2)} cities")
print(f"Failed:     {len(errors_v2)} cities")

if errors_v2:
    print()
    print("Failed cities:")
    for e in errors_v2:
        print(f"  {e['city']}: {e['reason']}")

### What improved?

- The pipeline ran all 15 cities without crashing
- The failed city was clearly identified with the reason
- All 14 successful records are in `results_v2`

But there is still a problem. Run the next cell to see it.

In [ ]:
# The free tier of OpenWeatherMap allows 60 calls per minute.
# 15 cities at full speed is fine today.
# But what if we needed 200 cities? Or ran this every 5 minutes?

# Let's simulate rapid requests and look at the rate limit headers

print("Making 5 rapid requests and reading rate limit signals...")
print()

test_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata"]

for city in test_cities:
    r = requests.get(BASE_URL, params={"q": city, "appid": API_KEY, "units": "metric"})

    # OpenWeatherMap does not expose X-RateLimit headers on the free tier,
    # but many APIs do. Here we show what to look for.
    remaining = r.headers.get("X-RateLimit-Remaining", "not in headers for this API")
    retry     = r.headers.get("Retry-After",            "not present")

    print(f"  {city:<12} status={r.status_code}  X-RateLimit-Remaining={remaining}  Retry-After={retry}")

print()
print("OpenWeatherMap does not send rate limit headers on the free tier.")
print("If you exceed 60 req/min, you will suddenly get 429 responses with no warning.")
print("The fix: add a deliberate pause between requests — do not rely on the API to tell you.")

---
### Exercise 1 — Error Handling

**Task:** Complete the `fetch_weather_safe` function below. It wraps a request in try/except and returns a `(record, error)` tuple. This is a simpler version than the full `fetch_weather` above — you only need to handle three cases: success, 404, and any other exception.

In [ ]:
def fetch_weather_safe(city, api_key):
    """
    Returns (record_dict, None) on success.
    Returns (None, error_string) on failure.
    """
    try:
        response = requests.get(
            BASE_URL,
            params  = {"q": city, "appid": api_key, "units": "metric"},
            timeout =10          # <-- how many seconds before giving up?
        )
        response.raise_for_status()             # <-- which method raises an error for 4xx/5xx?

        d = response.json()
        return {
            "city":     d["name"],
            "temp_c":   d["main"]["temp"],
            "humidity": d["main"]["humidity"]
        },None                    # <-- what is the second value on success?

    except requests.exceptions.HTTPError as e:
        code = e.response.status_code if e.response else "?"
        if code ==404:            # <-- which code means not found?
            return None, f"Not found: '{city}'"
        return None, f"HTTP error {code} for '{city}'"

    except requests.exceptions.RequestException:                    # <-- which exception class catches ALL requests errors?
        return None, f"Network error for '{city}'"


# Tests
r1, e1 = fetch_weather_safe("Jaipur", API_KEY)
print(f"Jaipur:      record={r1 is not None}  error={e1}")

r2, e2 = fetch_weather_safe("Xyznotacity", API_KEY)
print(f"Xyznotacity: record={r2}  error={e2}")

r3, e3 = fetch_weather_safe("Pune", "badkey000")
print(f"Bad key:     record={r3}  error={e3}")

print()
print(f"Correct (Jaipur succeeded):      {r1 is not None and e1 is None}")
print(f"Correct (fake city failed):      {r2 is None and e2 is not None}")
print(f"Correct (bad key failed):        {r3 is None and e3 is not None}")

<details>
<summary>Show solution</summary>

```python
def fetch_weather_safe(city, api_key):
    try:
        response = requests.get(
            BASE_URL,
            params  = {"q": city, "appid": api_key, "units": "metric"},
            timeout = 10
        )
        response.raise_for_status()
        d = response.json()
        return {
            "city":     d["name"],
            "temp_c":   d["main"]["temp"],
            "humidity": d["main"]["humidity"]
        }, None

    except requests.exceptions.HTTPError as e:
        code = e.response.status_code if e.response else "?"
        if code == 404:
            return None, f"Not found: '{city}'"
        return None, f"HTTP error {code} for '{city}'"

    except requests.exceptions.RequestException:
        return None, f"Network error for '{city}'"
```

</details>

---
# Part 4 — Fix 2: Rate Limiting

## What is rate limiting?

A rate limit is a cap on how many requests you can make in a given time window. APIs enforce them to:
- Prevent one user from consuming all the server's resources
- Ensure fair usage across all users
- Protect against accidental infinite loops in user code

When you exceed the limit, the API returns a **429 Too Many Requests** response. Depending on the API, it may come back immediately or lock you out for minutes.

---

## The three strategies

**Strategy 1: Fixed delay** — sleep a fixed number of seconds between every request. Simple. Predictable. Slightly slow.

**Strategy 2: Retry with backoff** — if you get a 429, wait and try again. Double the wait each time. This is called **exponential backoff**.

**Strategy 3: Read the Retry-After header** — some APIs tell you exactly how long to wait. If the header is there, use it.

A professional pipeline uses all three together.

---

### Whiteboard: exponential backoff

```
Attempt 1 fails (429) → wait 1 second  → retry
Attempt 2 fails (429) → wait 2 seconds → retry
Attempt 3 fails (429) → wait 4 seconds → retry
Attempt 4 fails (429) → wait 8 seconds → retry
Attempt 5 fails (429) → give up
```

Each wait is double the previous one. This is important: if many clients all get rate-limited at the same moment and all retry after exactly 1 second, they all hammer the server at the same time again. Random jitter + doubling spreads them out.

In [ ]:
# Strategy 1: Fixed delay
# The simplest approach. A small sleep between every request.

DELAY_BETWEEN_REQUESTS = 1.0   # seconds

def fetch_with_fixed_delay(city, api_key, delay=DELAY_BETWEEN_REQUESTS):
    """
    Fetches weather for a city, then sleeps for 'delay' seconds.
    The sleep happens AFTER the fetch — it is the gap before the next call.
    """
    record, error = fetch_weather(city, api_key)
    time.sleep(delay)   # polite pause
    return record, error


print("Testing fixed delay fetch:")
start = time.time()
for city in ["Mumbai", "Pune", "Nagpur"]:
    record, error = fetch_with_fixed_delay(city, API_KEY, delay=0.5)
    if record:
        print(f"  {record['city']}: {record['temp_c']}C")
elapsed = time.time() - start
print(f"\n3 cities fetched in {elapsed:.1f} seconds (0.5s delay each)")

In [ ]:
# Strategy 2: Retry with exponential backoff
# For when you DO hit a rate limit, or get a temporary 5xx error

def fetch_with_retry(city, api_key, max_retries=3):
    """
    Fetches weather with automatic retry on 429 or 5xx errors.
    Uses exponential backoff: waits 1s, then 2s, then 4s between retries.
    Returns (record, error) like fetch_weather.
    """
    wait = 1   # starting wait time in seconds

    for attempt in range(1, max_retries + 1):
        record, error = fetch_weather(city, api_key)

        # If it worked, return immediately
        if record:
            return record, None

        # If the error is a rate limit or server error, retry
        if error and ("429" in error or "5" in error[:5]):
            if attempt < max_retries:
                print(f"  Attempt {attempt} failed: {error}. Retrying in {wait}s...")
                time.sleep(wait)
                wait *= 2   # double the wait time
            else:
                print(f"  All {max_retries} attempts failed for '{city}'.")
                return None, error
        else:
            # For 404 or auth errors, retrying will not help — give up immediately
            return None, error

    return None, f"All retries exhausted for '{city}'"


# Test with a good city
record, error = fetch_with_retry("Kochi", API_KEY)
if record:
    print(f"Success: {record['city']}, {record['temp_c']}C")
else:
    print(f"Failed: {error}")

In [ ]:
# Strategy 3: A rate limiter that tracks your own request rate
# This is the most robust approach for production scripts

class RateLimiter:
    """
    Ensures you never exceed a given number of requests per minute.
    Call .wait() before each request.
    """

    def __init__(self, max_per_minute):
        self.min_interval   = 60.0 / max_per_minute  # seconds between requests
        self.last_call_time = 0.0

    def wait(self):
        """
        Sleeps if necessary to stay within the rate limit.
        """
        now     = time.time()
        elapsed = now - self.last_call_time

        if elapsed < self.min_interval:
            sleep_time = self.min_interval - elapsed
            time.sleep(sleep_time)

        self.last_call_time = time.time()


# Demonstration
limiter = RateLimiter(max_per_minute=30)   # 30 requests/min = 1 every 2 seconds

print("Making 3 requests at max 30 req/min (expect ~2s between each):")
for city in ["Delhi", "Chennai", "Kolkata"]:
    limiter.wait()             # enforce the rate before making the call
    t = time.time()
    record, error = fetch_weather(city, API_KEY)
    elapsed = time.time() - t
    status = record["temp_c"] if record else error
    print(f"  {city:<12} -> {status}  (request took {elapsed:.2f}s)")

---
### Exercise 2 — Rate Limiting

**Task:** Complete the pipeline loop below. It should:
1. Use the `RateLimiter` class to stay under 40 requests per minute
2. Call `fetch_with_retry` for each city (not the naive `requests.get`)
3. Append successes to `results` and failures to `errors`
4. Print a status line for each city

In [ ]:
results_v3 = []
errors_v3  = []

# Create a rate limiter — 40 requests per minute max
limiter = RateLimiter(max_per_minute=___)   # <-- what number?

for city in CITIES:
    ___.wait()                              # <-- which object's wait method?

    record, error = fetch_with_retry(___, API_KEY)   # <-- which variable is the city?

    if ___:                                 # <-- success condition
        results_v3.append(record)
        print(f"  OK    {city:<15} {record['temp_c']}C")
    else:
        errors_v3.append({"city": city, "reason": error})
        print(f"  FAIL  {city:<15} {___}")  # <-- print the error

print()
print(f"Successful: {len(results_v3)}")
print(f"Failed:     {len(errors_v3)}")
print(f"Correct (14 successes, 1 failure): {len(results_v3) == 14 and len(errors_v3) == 1}")

<details>
<summary>Show solution</summary>

```python
results_v3 = []
errors_v3  = []
limiter    = RateLimiter(max_per_minute=40)

for city in CITIES:
    limiter.wait()
    record, error = fetch_with_retry(city, API_KEY)
    if record:
        results_v3.append(record)
        print(f"  OK    {city:<15} {record['temp_c']}C")
    else:
        errors_v3.append({"city": city, "reason": error})
        print(f"  FAIL  {city:<15} {error}")
```

</details>

---
# Part 5 — Fix 3: Parsing API Responses into CSV and JSON

## The problem with just printing results

After all the fetching, our results are sitting in a Python list in memory. The moment this notebook session ends, they are gone. Worse, the manager who needs the daily weather report cannot open a Python list — they need a file.

We need to **persist** the results. That means writing them to disk in a usable format.

---

## Two output formats — and when to use each

We covered both formats in Session 3.1/3.2. Here is a reminder of when to choose each for pipeline output:

| Format | Best for | Limitations |
|---|---|---|
| CSV | Flat data, spreadsheet consumers, simple analysis | No nested data, no types |
| JSON | Nested data, other programs, APIs, dashboards | Less convenient for spreadsheets |

Our weather records are flat (every record has the same fields), so CSV works perfectly. But we will save both — the manager gets the CSV, the dashboard system gets the JSON.

---

## The save-as-you-go pattern

There are two ways to write results to disk:

**Option A: Collect everything, write at the end**
- Simpler code
- If the script crashes at record 12, records 1–11 are lost

**Option B: Write each record immediately after fetching it**
- Slightly more complex
- If the script crashes at record 12, records 1–11 are already safely on disk

For a production pipeline, Option B is always the right choice. We will implement both so you can compare them.

In [ ]:
# Option A: Collect everything, then write at the end
# Using results_v3 from the previous section

FIELDNAMES = ["city", "country", "temp_c", "feels_like", "humidity", "wind_ms", "description", "fetched_at"]


def save_to_csv(records, filename):
    """Writes a list of weather records to a CSV file."""
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        writer.writeheader()
        writer.writerows(records)
    print(f"Saved {len(records)} records to '{filename}'")


def save_to_json(records, errors, filename):
    """Writes a structured output (records + errors + metadata) to JSON."""
    payload = {
        "run_timestamp": datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC"),
        "total_cities":  len(records) + len(errors),
        "successful":    len(records),
        "failed":        len(errors),
        "records":       records,
        "errors":        errors
    }
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)
    print(f"Saved {len(records)} records + {len(errors)} errors to '{filename}'")


# Save the results from v3
save_to_csv(results_v3,  "weather_report.csv")
save_to_json(results_v3, errors_v3, "weather_report.json")

print()

# Verify: read back and show the first 3 rows
print("CSV preview:")
with open("weather_report.csv", "r") as f:
    reader = csv.DictReader(f)
    for i, row in enumerate(reader):
        if i >= 3: break
        print(f"  {row['city']:<15} {row['temp_c']:>5}C  {row['humidity']:>3}%  {row['description']}")

In [ ]:
# Option B: Write each record to disk immediately after fetching
# Open the files once, keep writing into them, close at the end

def run_pipeline_streaming(cities, api_key, csv_path, json_path, max_per_minute=40):
    """
    Full pipeline that writes results to disk as they arrive.
    A crash mid-run will not lose already-written records.
    """
    limiter   = RateLimiter(max_per_minute=max_per_minute)
    errors    = []
    record_count = 0

    print(f"Starting pipeline: {len(cities)} cities -> '{csv_path}' and '{json_path}'")
    print(f"Rate limit: {max_per_minute} requests/min")
    print()

    # Open the CSV file once — keep it open while we loop
    with open(csv_path, "w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=FIELDNAMES)
        writer.writeheader()

        for city in cities:
            limiter.wait()

            record, error = fetch_with_retry(city, api_key)

            if record:
                writer.writerow(record)    # written to disk immediately
                csv_file.flush()           # force the OS to flush to disk now
                record_count += 1
                print(f"  OK    {city:<15} {record['temp_c']}C  ({record['description']})")
            else:
                errors.append({"city": city, "reason": error})
                print(f"  FAIL  {city:<15} {error}")

    # JSON is written at the end (it is a single object — cannot be streamed as simply)
    with open(json_path, "w", encoding="utf-8") as jf:
        # We need to re-read the CSV to build the JSON (since we streamed the CSV)
        with open(csv_path, "r", encoding="utf-8") as cf:
            records = list(csv.DictReader(cf))

        json.dump({
            "run_timestamp": datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC"),
            "successful":    record_count,
            "failed":        len(errors),
            "records":       records,
            "errors":        errors
        }, jf, indent=2)

    print()
    print(f"Pipeline complete: {record_count} saved, {len(errors)} failed")
    print(f"CSV  -> {csv_path}  ({os.path.getsize(csv_path)} bytes)")
    print(f"JSON -> {json_path}  ({os.path.getsize(json_path)} bytes)")


run_pipeline_streaming(
    cities       = CITIES,
    api_key      = API_KEY,
    csv_path     = "weather_streaming.csv",
    json_path    = "weather_streaming.json",
    max_per_minute = 40
)

---
### Exercise 3 — Parsing and Saving

**Task:** Complete the `save_error_log` function below. A good pipeline not only saves the successful records — it also saves the failures to a separate file so they can be reviewed and retried later.

The function should:
1. Accept the errors list and a filename
2. Write the errors to a CSV with columns: `city`, `reason`, `logged_at`
3. Add the current UTC timestamp to each row as `logged_at`
4. Return the number of errors written

In [ ]:
def save_error_log(errors, filename):
    """
    Saves a list of error dicts to a CSV file.
    Each error has 'city' and 'reason'.
    Adds a 'logged_at' timestamp column.
    Returns the number of errors written.
    """
    timestamp  = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")
    fieldnames = [___, ___, ___]     # <-- three column names

    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=___)
        writer.___()

        for error in errors:
            writer.writerow({
                "city":      error[___],
                "reason":    error[___],
                "logged_at": ___        # <-- the timestamp variable
            })

    return len(errors)


count = save_error_log(errors_v3, "error_log.csv")
print(f"Error log written: {count} entries")
print()
print("Contents of error_log.csv:")
with open("error_log.csv", "r") as f:
    print(f.read())

print(f"Correct (1 error written): {count == 1}")
print(f"Correct (Coimbatorr logged): {'Coimbatorr' in open('error_log.csv').read()}")

<details>
<summary>Show solution</summary>

```python
def save_error_log(errors, filename):
    timestamp  = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")
    fieldnames = ["city", "reason", "logged_at"]

    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for error in errors:
            writer.writerow({
                "city":      error["city"],
                "reason":    error["reason"],
                "logged_at": timestamp
            })

    return len(errors)
```

</details>

---
# Part 6 — The Finished Pipeline

## Before and after

Let's look at what we started with and what we built:

**The naive pipeline (12 lines):**
```python
results = []
for city in CITIES:
    response = requests.get(BASE_URL, params={"q": city, "appid": API_KEY, "units": "metric"})
    data = response.json()
    record = {"city": data["name"], "temp_c": data["main"]["temp"], ...}
    results.append(record)
```

**Problems:** crashes on any error, no retry, no rate limiting, no output files, results lost on crash.

**The finished pipeline — all fixes combined.** Read through every line and trace it back to the fix that motivated it.

In [ ]:
def run_weather_pipeline(
    cities,
    api_key,
    csv_output   = "weather_final.csv",
    json_output  = "weather_final.json",
    error_log    = "errors_final.csv",
    max_per_min  = 40,
    max_retries  = 3
):
    """
    Complete weather data fetching pipeline.

    Features:
      - Error handling: each city failure is caught, logged, and skipped
      - Rate limiting: respects the API's request limits automatically
      - Retry logic: temporary failures are retried with exponential backoff
      - Streaming writes: CSV records are written to disk immediately
      - Dual output: saves both CSV (for spreadsheet users) and JSON (for systems)
      - Error log: failed cities are saved separately for later review
    """

    # ----------------------------------------------------------------
    # Initialise
    # ----------------------------------------------------------------
    limiter      = RateLimiter(max_per_minute=max_per_min)
    errors       = []
    record_count = 0
    run_start    = datetime.utcnow()

    print("=" * 60)
    print(f"WEATHER PIPELINE  |  {run_start.strftime('%Y-%m-%d %H:%M UTC')}")
    print(f"Cities: {len(cities)}  |  Rate limit: {max_per_min}/min  |  Max retries: {max_retries}")
    print(f"Outputs: {csv_output}, {json_output}, {error_log}")
    print("=" * 60)

    # ----------------------------------------------------------------
    # Fetch and stream-write to CSV
    # ----------------------------------------------------------------
    with open(csv_output, "w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=FIELDNAMES)
        writer.writeheader()

        for city in cities:

            # Rate limiting — pause if we are going too fast
            limiter.wait()

            # Fetch with retry and error handling
            record, error = fetch_with_retry(city, api_key, max_retries=max_retries)

            if record:
                # Write immediately — do not wait until the end
                writer.writerow(record)
                csv_file.flush()   # flush to disk right now
                record_count += 1
                print(f"  OK    {city:<15} {record['temp_c']:>5.1f}C  {record['humidity']:>3}%  {record['description']}")
            else:
                errors.append({"city": city, "reason": error})
                print(f"  FAIL  {city:<15} {error}")

    # ----------------------------------------------------------------
    # Write JSON output
    # ----------------------------------------------------------------
    with open(csv_output, "r", encoding="utf-8") as cf:
        all_records = list(csv.DictReader(cf))

    run_end   = datetime.utcnow()
    duration  = (run_end - run_start).total_seconds()

    with open(json_output, "w", encoding="utf-8") as jf:
        json.dump({
            "run_start":    run_start.strftime("%Y-%m-%d %H:%M:%S UTC"),
            "run_end":      run_end.strftime("%Y-%m-%d %H:%M:%S UTC"),
            "duration_sec": round(duration, 1),
            "successful":   record_count,
            "failed":       len(errors),
            "records":      all_records,
            "errors":       errors
        }, jf, indent=2)

    # ----------------------------------------------------------------
    # Write error log
    # ----------------------------------------------------------------
    if errors:
        save_error_log(errors, error_log)

    # ----------------------------------------------------------------
    # Summary
    # ----------------------------------------------------------------
    print()
    print("=" * 60)
    print(f"PIPELINE COMPLETE")
    print(f"  Duration:    {duration:.1f} seconds")
    print(f"  Successful:  {record_count} cities")
    print(f"  Failed:      {len(errors)} cities")
    print(f"  CSV output:  {csv_output}  ({os.path.getsize(csv_output):,} bytes)")
    print(f"  JSON output: {json_output}  ({os.path.getsize(json_output):,} bytes)")
    if errors:
        print(f"  Error log:   {error_log}  ({os.path.getsize(error_log):,} bytes)")
    print("=" * 60)

    return all_records, errors


# Run the finished pipeline
records, errors = run_weather_pipeline(
    cities      = CITIES,
    api_key     = API_KEY,
    max_per_min = 40
)

In [ ]:
# Verify the output files

print("CSV output (first 5 rows):")
print(f"  {'City':<15} {'Temp':>6}  {'Humidity':>8}  Description")
print("  " + "-" * 55)
with open("weather_final.csv", "r") as f:
    reader = csv.DictReader(f)
    for i, row in enumerate(reader):
        if i >= 5: break
        print(f"  {row['city']:<15} {row['temp_c']:>5}C  {row['humidity']:>7}%  {row['description']}")

print()
print("JSON output (structure):")
with open("weather_final.json", "r") as f:
    j = json.load(f)

print(f"  run_start:    {j['run_start']}")
print(f"  duration_sec: {j['duration_sec']}")
print(f"  successful:   {j['successful']}")
print(f"  failed:       {j['failed']}")
print(f"  records:      {len(j['records'])} items")
print(f"  errors:       {len(j['errors'])} items")

if j['errors']:
    print()
    print("Errors logged in JSON:")
    for e in j['errors']:
        print(f"  {e['city']}: {e['reason']}")

---
## Final Exercise — Build Your Own Pipeline

This is the session's capstone. You will build a complete pipeline from scratch — but for a different data source.

**Scenario:** Instead of weather data, you are now building a pipeline that fetches information about GitHub repositories from a list of repository names in the `python` organisation, and saves the results to CSV and JSON files.

The GitHub API endpoint for a single repo is:
```
GET https://api.github.com/repos/{owner}/{repo}
```

No API key needed. The response includes fields like `name`, `stargazers_count`, `language`, `description`, `open_issues_count`, and `updated_at`.

Complete all four cells below.

In [ ]:
# Step 1: The fetch function
# Complete fetch_repo() so it returns (record, error) like fetch_weather does.

GITHUB_REPOS = [
    "cpython", "mypy", "black", "pip", "peps",
    "typeshed", "pyperclip", "notarealrepo99", "packaging"
]

GITHUB_FIELDNAMES = ["name", "stars", "language", "open_issues", "description", "updated_at"]

def fetch_repo(repo_name, owner="python"):
    """
    Fetches metadata for a single GitHub repository.
    Returns (record_dict, None) on success, (None, error_string) on failure.
    """
    url     = f"https://api.github.com/repos/{owner}/{repo_name}"
    headers = {"Accept": "application/vnd.github+json"}

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        d = response.json()
        return {
            "name":        d["name"],
            "stars":       d[___],             # <-- field for star count
            "language":    d.get("language") or "N/A",
            "open_issues": d[___],             # <-- field for open issue count
            "description": (d.get("description") or "")[:80],
            "updated_at":  d[___]              # <-- field for last updated time
        }, None

    except requests.exceptions.HTTPError as e:
        code = e.response.status_code if e.response else "?"
        if code == 404:
            return None, f"Repo not found: '{repo_name}'"
        return None, f"HTTP {code} for '{repo_name}'"

    except requests.exceptions.RequestException as e:
        return None, f"Network error for '{repo_name}': {e}"


# Quick test
r, e = fetch_repo("cpython")
print(f"cpython: stars={r['stars'] if r else None}  error={e}")

r2, e2 = fetch_repo("notarealrepo99")
print(f"notarealrepo99: record={r2}  error={e2}")

In [ ]:
# Step 2: The pipeline loop
# Complete the loop to fetch all repos, write to CSV as you go, and collect errors.

github_results = []
github_errors  = []
limiter_gh     = RateLimiter(max_per_minute=30)  # GitHub allows 60/hr without auth; be cautious

with open("github_repos.csv", "w", newline="", encoding="utf-8") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=___)
    writer.writeheader()

    for repo_name in GITHUB_REPOS:
        ___.wait()                             # <-- rate limiter

        record, error = fetch_repo(___)

        if record:
            writer.writerow(___)
            csv_file.flush()
            github_results.append(record)
            print(f"  OK    {repo_name:<20} stars={record['stars']:>6,}")
        else:
            github_errors.append({"city": repo_name, "reason": error})
            print(f"  FAIL  {repo_name:<20} {error}")

print()
print(f"Fetched: {len(github_results)}  Failed: {len(github_errors)}")

In [ ]:
# Step 3: Save to JSON and write the error log

# Save JSON output
with open("github_repos.json", "w", encoding="utf-8") as jf:
    json.dump({
        "fetched_at":  datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC"),
        "successful":  len(github_results),
        "failed":      len(github_errors),
        "repos":       ___,     # <-- which list?
        "errors":      ___      # <-- which list?
    }, jf, indent=___)

# Save error log if there are any failures
if github_errors:
    save_error_log(github_errors, "github_errors.csv")

print("Files written:")
for fname in ["github_repos.csv", "github_repos.json", "github_errors.csv"]:
    if os.path.exists(fname):
        print(f"  {fname}  ({os.path.getsize(fname):,} bytes)")

In [ ]:
# Step 4: Print a sorted summary table
# Sort by stars descending and display the results

sorted_results = sorted(github_results, key=lambda r: int(r[___]), reverse=True)  # <-- sort key

print("GITHUB REPOSITORY REPORT — python organisation")
print("=" * 75)
print(f"  {'Repo':<22} {'Stars':>8}  {'Language':<12}  {'Issues':>7}  Description")
print("  " + "-" * 73)

for r in sorted_results:
    desc = r['description'][:35] + '...' if len(r['description']) > 35 else r['description']
    print(f"  {r['name']:<22} {int(r['stars']):>8,}  {r['language']:<12}  {int(r['open_issues']):>7,}  {desc}")

print()
print(f"Correct (cpython has most stars): {sorted_results[0]['name'] == 'cpython' if sorted_results else False}")
print(f"Correct (notarealrepo99 not in results): {all(r['name'] != 'notarealrepo99' for r in sorted_results)}")

<details>
<summary>Show solutions for all four steps</summary>

**Step 1:**
```python
"stars":       d["stargazers_count"],
"open_issues": d["open_issues_count"],
"updated_at":  d["updated_at"]
```

**Step 2:**
```python
writer = csv.DictWriter(csv_file, fieldnames=GITHUB_FIELDNAMES)
limiter_gh.wait()
record, error = fetch_repo(repo_name)
writer.writerow(record)
```

**Step 3:**
```python
"repos":  github_results,
"errors": github_errors
}, jf, indent=2)
```

**Step 4:**
```python
sorted_results = sorted(github_results, key=lambda r: int(r["stars"]), reverse=True)
```

</details>

---
## Session Summary

### The evolution of our pipeline

| Version | What we added | Problem it solved |
|---|---|---|
| Naive | Basic loop with `requests.get` | Nothing — crashes on any error |
| v2 | Error handling with try/except | One failure no longer kills the run |
| v3 | Rate limiting with `RateLimiter` | Prevents 429 rate limit errors |
| v3 | Retry with exponential backoff | Recovers from temporary failures |
| Final | Streaming CSV writes + JSON + error log | No data loss, full audit trail |

---

### Key concepts

**Error handling**
- Wrap every request in `try/except`
- Catch `HTTPError` for server-side failures, `RequestException` for network failures
- Always use `timeout=` to prevent hanging forever
- Return `(record, None)` on success and `(None, error_string)` on failure — never raise from inside a pipeline loop

**Rate limiting**
- Never assume the API will warn you before cutting you off
- A `RateLimiter` class tracks your own request rate and sleeps as needed
- `time.sleep()` between requests is the simplest form — good enough for small pipelines
- Exponential backoff: double the wait time on each consecutive failure — spreads out retries

**Parsing and saving**
- Write CSV records to disk immediately using `csv_file.flush()` — do not buffer in memory
- JSON output wraps the records with metadata (timestamps, counts, errors)
- Always save an error log for failed items — so they can be retried without re-running everything

---

### What comes next

You now have the skills to fetch real data from any API and save it to disk reliably. The next modules move into **Pandas** — where you will load these CSV and JSON files into dataframes and start performing analysis: aggregations, filtering, sorting, visualisation. The data you fetched today is exactly the kind of data you will be analysing.

---

### Quick reference — the complete pipeline pattern

```python
import requests, csv, json, time
from datetime import datetime

# 1. Rate limiter
class RateLimiter:
    def __init__(self, max_per_minute):
        self.min_interval = 60.0 / max_per_minute
        self.last_call    = 0.0
    def wait(self):
        elapsed = time.time() - self.last_call
        if elapsed < self.min_interval:
            time.sleep(self.min_interval - elapsed)
        self.last_call = time.time()

# 2. Fetch function with error handling
def fetch_item(identifier, api_key):
    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        return parse(r.json()), None
    except requests.exceptions.HTTPError as e:
        return None, f"HTTP {e.response.status_code}"
    except requests.exceptions.RequestException as e:
        return None, str(e)

# 3. Pipeline loop — stream-write as you go
limiter = RateLimiter(max_per_minute=40)
with open("output.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
    writer.writeheader()
    for item in items:
        limiter.wait()
        record, error = fetch_item(item, API_KEY)
        if record:
            writer.writerow(record)
            f.flush()
        else:
            errors.append({"item": item, "reason": error})
```